#Binary Anomaly Segmentation for Autonomous Driving

## Project Overview
This project implements a pixel-wise binary anomaly detection system. Unlike standard semantic segmentation, which assigns a specific category label to every pixel, our objective is to distinguish between "In-Distribution" (ID) content, originating from the Cityscapes dataset, and "Out-of-Distribution" (OOD) content, represented by anomalous objects (e.g., COCO objects) pasted into the driving scenes.

### 1. Conceptual Framework: From Multi-class to Binary Anomaly
The core challenge lies in transforming a model pre-trained on a fixed set of classes into a general-purpose anomaly detector. The fundamental reasoning is as follows: a pixel is considered an anomaly if no neural query can confidently assign it to a known semantic category.The architecture utilizes a Query-based Mask Transformer (EoMT). Each query within the decoder produces two distinct outputs:
1. Class Logits: Identifying what the query represent.
2. Mask Logits: Identifying where the query is spatially located.

### 2. Algorithmic Implementation of the Outlier Score
To compute the probability of a pixel being anomalous, we follow a multi-step probabilistic fusion:

**A. Class Probability Estimation**

We apply a Softmax function over the class logits to obtain a probability distribution for each query.$$class\_prob = \text{Softmax}(class\_logits, \text{dim}=-1)$$

**B. Maximum Known Confidence**

We determine the query's certainty regarding known categories. If the model is unsure about any known class, the probability of it being a "known" object decreases.$$max\_known\_prob = \max_{c \in \text{known}} (class\_prob)$$

**C. Spatial Localization**

A Sigmoid function is applied to the mask logits to determine the probability of a pixel belonging to the spatial extent of a query.$$mask\_prob = \sigma(mask\_logits)$$

**D. Pixel-wise Knowledge Fusion**

A pixel is defined as known only if there exists at least one query that both covers the pixel spatially and is highly confident in its semantic classification. We aggregate this across all queries using the maximum operator:$$pixel\_known\_conf = \max_{q} (mask\_prob_{q} \times max\_known\_prob_{q})$$

**E. Final Anomaly Scoring**

The final Out-of-Distribution (OOD) score is the complement of the knowledge confidence.$$OOD\_score = 1 - pixel\_known\_conf$$

# Usage Instructions

## Environment Preparation
- Mount Storage: Execute the setup cell to mount Google Drive for access to checkpoints and the prepared Outlier Exposure (OE) dataset.

- Authentication: Log in to Weights & Biases (WandB) to track training metrics such as AUROC and AUPRC.

## Training Workflow
- Configuration: Define hyperparameters in the AnomalyConfig class (Batch Size, Learning Rate, Epochs).

- Model Loading: The script loads the pre-trained EoMT weights, replaces the 19-class head with a binary head, and freezes the ViT encoder to preserve robust features.

- Optimization: The trainer utilizes a combined Dice + Focal Loss to address the class imbalance between rare anomalies and common background pixels.

## Evaluation
The model logs three key outputs to the dashboard for qualitative assessment:

- Input Image: The original RGB frame.

- Anomaly Map: The heatmap of predicted outlier scores.

- Ground Truth: The binary mask of the actual anomalous objects.

In [ ]:
# ============================================================
# BINARY ANOMALY DETECTION - ENVIRONMENT SETUP
# ============================================================
# Description:
#   Initial setup script for Google Colab environment.
#   Mounts Google Drive for dataset access and authenticates
#   Weights & Biases for experiment tracking.
# ============================================================

import sys
import os

# 1. Mount Google Drive
# This allows access to the persistent storage where datasets and checkpoints are saved.
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("Warning: Google Colab environment not detected. Skipping Drive mount.")

# 2. Login to Weights & Biases
# Authenticates the session to log metrics and artifacts to the cloud dashboard.
try:
    import wandb
    print("Authenticating Weights & Biases...")
    wandb.login()
except ImportError:
    print("Error: wandb library not installed. Please install it via 'pip install wandb'.")

print("Environment setup completed successfully.")

Mounting Google Drive...


wandb: WARNING Unable to verify login in offline mode.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Authenticating Weights & Biases...
Environment setup completed successfully.


In [ ]:
!pip install -q torch torchvision lightning transformers timm wandb torchmetrics

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import sys
import math
import yaml
import importlib
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# PyTorch Lightning & Metrics
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from torchmetrics import AUROC, AveragePrecision
import wandb

# Torchvision
from torchvision import transforms
from torchvision import tv_tensors

# Custom Modules (Environment Specific)
# Note: Adjust the path below according to the specific deployment environment
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/AML_Project/MaskArchitectureAnomaly_CourseProject/eomt"
sys.path.insert(0, PROJECT_ROOT)

from training.mask_classification_semantic import MaskClassificationSemantic
from training.mask_classification_loss import MaskClassificationLoss
from datasets.transforms import Transforms
from models.eomt import EoMT, ScaleBlock
from models.vit import ViT

print("EoMT modules imported successfully.")

# ============================================================
# CONFIGURATION
# ============================================================
class AnomalyConfig:
    """
    Configuration for binary anomaly detection training.
    Contains all hyperparameters and paths for the experiment.
    """
    # Model architecture configuration
    img_size = (640, 640)

    # Data loading configuration
    batch_size = 8
    num_workers = 4
    num_gpus = -1  # Use all available GPUs (-1 for auto-detect))

    # Checkpoint paths
    base_model_checkpoint = os.path.join(PROJECT_ROOT, "trained_model/epoch_106-step_19902_eomt.ckpt")

    # Path to pre-prepared Outlier Exposure dataset
    prepared_oe_path = "/content/drive/MyDrive/Colab Notebooks/AML_Project/dataset_preparation/Cityscapes_and_COCO/dataset_oe"

    # Expected structure:
    # prepared_oe_dataset/
    # ├── train/
    # │   ├── images/           (RGB images with pasted COCO objects)
    # │   └── anomaly_masks/    (Binary masks: 0=normal, 1=anomaly)

    # Training hyperparameters
    max_epochs = 10
    lr = 1e-4 * (math.sqrt(num_gpus) if num_gpus > 0 else 1) # Scaled LR
    weight_decay = 0.01

    # Model strategy
    freeze_encoder = True  # Freeze Vision Transformer encode

    # Output directory
    output_dir = "/content/drive/MyDrive/Colab Notebooks/AML_Project/fine-tuning_extension/custom_checkpoint_final_extension"

config = AnomalyConfig()

# ============================================================
# LOSS FUNCTION: BinaryAnomalyLoss
# ===========================================================
class BinaryAnomalyLoss(nn.Module):
    """
    Composite loss function for Binary Anomaly Detection.
    Combines Dice Loss and Focal Loss.
    """
    def __init__(self, dice_weight=5.0):
        super().__init__()
        self.dice_weight = dice_weight

    def dice_loss(self, pred: torch.Tensor, target: torch.Tensor, smooth: float = 1e-6) -> torch.Tensor:
        """Computes Dice Loss for segmentation."""
        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (pred * target).sum()
        dice_coeff = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
        return 1 - dice_coeff

    def focal_loss(self, pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-7, gamma: int = 2) -> torch.Tensor:
        """Computes Focal Loss to address class imbalance."""
        pred = torch.clamp(pred, min=eps, max=1 - eps)
        loss = - (target * (1 - pred) ** gamma * torch.log(pred) +
                  (1 - target) * pred ** gamma * torch.log(1 - pred))
        return loss.mean()

    def forward(self, anomaly_scores, anomaly_masks):
        """
        Args:
            anomaly_scores: Predicted probabilities [B, H, W]
            anomaly_masks: Ground truth binary masks [B, H, W]
        """
        #Dice Loss component
        loss_dice = self.dice_loss(anomaly_scores, anomaly_masks)

        #Numerical stability
        anomaly_scores = torch.clamp(anomaly_scores, min=1e-7, max=1-1e-7)

        #Focal Loss component
        focal_loss = self.focal_loss(anomaly_scores, anomaly_masks)


        # weighted combination
        total_loss =  (self.dice_weight * loss_dice) + 2 * focal_loss

        return total_loss

# ============================================================
# DATASET PreparedOEDataset
# ============================================================
class PreparedOEDataset(torch.utils.data.Dataset):
    """
    Dataset loader for pre-prepared Outlier Exposure (OE) dataset.
    Assumes images have pasted COCO objects on Cityscapes backgrounds.
    """

    def __init__(self, root_path: str, split: str = 'train', transforms=None, subset_size: Optional[int] = None):
        self.root = Path(root_path)
        self.img_dir = self.root / split / "images"
        self.mask_dir = self.root / split / "anomaly_masks"
        self.transforms = transforms

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image directory not found: {self.img_dir}")
        if not self.mask_dir.exists():
            raise FileNotFoundError(f"Anomaly mask directory not found: {self.mask_dir}")

        self.images = sorted(
            list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.png"))
        )

        if subset_size:
            self.images = self.images[:subset_size]

        print(f"Loaded {len(self.images)} {split} samples from {root_path}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        #Load a single sample from the dataset.

        img_path = self.images[idx]

        # Load and convert image to RGB
        img = Image.open(img_path).convert("RGB")
        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float()  # [3, H, W]

        # Load binary anomaly mask
        mask_path = self.mask_dir / img_path.name.replace('.jpg', '.png')
        anomaly_mask = Image.open(mask_path).convert("L")
        anomaly_mask = torch.from_numpy(np.array(anomaly_mask)).float()

        # Normalize mask to binary {0, 1}
        if anomaly_mask.max() > 1:
            anomaly_mask = (anomaly_mask > 128).float()

        # Apply resizing if transforms are present
        if self.transforms and hasattr(self.transforms, 'img_size'):
            from torchvision.transforms.v2 import functional as F_v2
            target_h, target_w = self.transforms.img_size

            img = F_v2.resize(
                img,
                (target_h, target_w),
                interpolation=F_v2.InterpolationMode.BILINEAR
            )
            anomaly_mask = F_v2.resize(
                anomaly_mask.unsqueeze(0),
                (target_h, target_w),
                interpolation=F_v2.InterpolationMode.NEAREST
            ).squeeze(0)

        target = {'anomaly_mask': anomaly_mask}
        return img, target

# ============================================================
# LIGHTNING MODULE
# ============================================================
class AnomalyDetectionModuleStreamingMetrics(MaskClassificationSemantic):
    """
    Lightning Module for Anomaly Detection.
    Implements streaming metrics calculation to manage memory efficiency.
    """

    def __init__(self, *args, freeze_encoder: bool = True, **kwargs):
        super().__init__(*args, **kwargs)

        if freeze_encoder:
            for param in self.network.encoder.parameters():
                param.requires_grad = False
            print("Encoder frozen.")

        self.criterion = BinaryAnomalyLoss(dice_weight=5.0)

        # Streaming metrics
        self.val_auroc = AUROC(task="binary")
        self.val_auprc = AveragePrecision(task="binary")


    def _on_eval_end_semantic(self, log_prefix):
        """
        Override parent method to prevent standard mIoU calculation
        which is not applicable for this anomaly detection task.
        """
        pass

    def _predict(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass to generate anomaly scores.
        """
        mask_logits_per_block, class_logits_per_block = self.network(x)
        mask_logits = mask_logits_per_block[-1]   # [B, Q, 160, 160]
        class_logits = class_logits_per_block[-1] # [B, Q, C]

        # 1. Immediate Upsampling of logits
        mask_logits_high = F.interpolate(
            mask_logits,
            size=config.img_size,
            mode='bilinear',
            align_corners=False
        ) # Result: [B, Q, 640, 640]

        # 2. Probability calculation
        mask_prob = torch.sigmoid(mask_logits_high)
        class_prob = torch.softmax(class_logits, dim=-1)

        # Extract probability of known classes
        max_knownprob = class_prob[..., 0]

        # 3. Final Fusion
        pixel_known_conf = torch.max(
            mask_prob * max_knownprob[:, :, None, None],
            dim=1
        )[0]

        # Anomaly score is the inverse of known confidence
        anomaly_score = 1 - pixel_known_conf
        return anomaly_score

    def training_step(self, batch, batch_idx):
        imgs, targets = batch
        anomaly_scores = self._predict(imgs)
        anomaly_masks = torch.stack([t['anomaly_mask'] for t in targets])

        loss = self.criterion(anomaly_scores, anomaly_masks)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_start(self):
        self.val_auroc.reset()
        self.val_auprc.reset()


    def validation_step(self, batch, batch_idx):
        imgs, targets = batch
        anomaly_scores = self._predict(imgs)
        anomaly_masks = torch.stack([t['anomaly_mask'] for t in targets]).float()

        # Loss calculation
        loss = self.criterion(anomaly_scores, anomaly_masks)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True, batch_size=imgs.shape[0])

        # Metric updates (using spatial downsampling to reduce memory load)
        stride = 4
        sampled_scores = anomaly_scores[:, ::stride, ::stride].flatten()
        sampled_masks = anomaly_masks[:, ::stride, ::stride].flatten().long()

        self.val_auroc.update(sampled_scores, sampled_masks)
        self.val_auprc.update(sampled_scores, sampled_masks)

        # Log visualization to WandB periodically
        if batch_idx % 50 == 0:
            pred_viz = (anomaly_scores[0].detach().unsqueeze(0) * 255).cpu().numpy().astype(np.uint8)
            mask_viz = (anomaly_masks[0].detach().unsqueeze(0) * 255).cpu().numpy().astype(np.uint8)
            img_viz = (imgs[0].detach().permute(1, 2, 0) * 255).cpu().numpy().astype(np.uint8)

            if self.logger:
                self.logger.experiment.log({
                    "val_samples": [
                        wandb.Image(img_viz, caption="Input"),
                        wandb.Image(pred_viz, caption="Prediction"),
                        wandb.Image(mask_viz, caption="Ground Truth")
                    ]
                })
        return loss

    def on_validation_epoch_end(self):
        try:
            auroc = self.val_auroc.compute()
            auprc = self.val_auprc.compute()

            self.log("val/auroc", auroc, prog_bar=True, sync_dist=True)
            self.log("val/auprc", auprc, prog_bar=True, sync_dist=True)

            print(f"Validation Results - AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}")

        except Exception as e:
            print(f"Error computing metrics: {e}")

def collate_fn(batch):
    """
    Custom collate function for batching.
    Stacks images but keeps targets as a list of dictionaries.
    """
    imgs = torch.stack([b[0] for b in batch])
    targets = [b[1] for b in batch]
    return imgs, targets

print("Classes defined successfully.")

EoMT modules imported successfully.
Classes defined successfully.


In [ ]:
# ============================================================
# SETUP AND TRAINING
# ============================================================

print("="*70)
print("BINARY ANOMALY DETECTION - FINE-TUNING PRE-TRAINED EOMT")
print("="*70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Step 1: Load pre-trained model
# ------------------------------------------------------------
print("\nStep 1: Loading pre-trained EoMT model...")

try:
    # 1.1 Load Model Configuration
    # Adjust path to the specific YAML config used during pre-training
    config_path = os.path.join(PROJECT_ROOT, "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml")
    if not os.path.exists(config_path):
        # Fallback relative path
        config_path = "eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

    with open(config_path) as f:
        yaml_config = yaml.safe_load(f)

    # 1.2 Instantiate Encoder (Vision Transformer)
    # Extract encoder configuration from YAML
    encoder_cfg = yaml_config["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
    encoder = encoder_cls(img_size=config.img_size, **encoder_cfg.get("init_args", {}))

    # 1.3 Instantiate Network (EoMT)
    network_cfg = yaml_config["model"]["init_args"]["network"]
    network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

    # Filter arguments to avoid passing 'encoder' twice
    network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

    network = network_cls(
        masked_attn_enabled=False,
        num_classes=1, # Overriding for binary anomaly detection
        encoder=encoder,
        **network_kwargs,
    )

   # 1.4 Initialize Lightning Module
    # This wraps the network with our specific training/validation logic
    model = AnomalyDetectionModuleStreamingMetrics(
        network=network,
        img_size=config.img_size,
        attn_mask_annealing_enabled=False,
        freeze_encoder=config.freeze_encoder,
        num_classes=1,
    )

    # 1.5 Load Pre-trained Weights
    checkpoint = torch.load(config.base_model_checkpoint, map_location=device)
    state_dict = checkpoint.get("state_dict", checkpoint)

    # Clean state_dict keys to match current model structure
    state_dict = {k.replace("network.", "").replace("model.", ""): v for k, v in state_dict.items() if "pos_embed" not in k}

    # Remove the old classification head weights
    # The pre-trained model has 19 classes (Cityscapes), but we need 1 class (Anomaly).
    # Removing these keys allows the new head to be initialized with random weights for fine-tuning.
    keys_to_remove = ["class_head.weight", "class_head.bias"]
    for k in keys_to_remove:
        if k in state_dict:
            print(f"Ignoring pre-trained weights for: {k} (Expected shape mismatch)")
            del state_dict[k]

    # Load the filtered weights
    network.load_state_dict(state_dict, strict=False)

    # Move to device and ensure float32 precision
    model.to(device)
    model = model.float()

    print(f"Success: Model loaded from {config.base_model_checkpoint}")

except Exception as e:
    print(f"Error: Failed to load model: {e}")
    sys.exit(1)

# ------------------------------------------------------------
# Step 2: Load Datasets
# ------------------------------------------------------------
print("\nStep 2: Loading prepared Outlier Exposure datasets...")

transforms_obj = Transforms(
    img_size=config.img_size,
    color_jitter_enabled=True,
    scale_range=(0.5, 2.0),
)

try:
      # Load the full dataset (images and masks)
    full_dataset = PreparedOEDataset(
        config.prepared_oe_path,
        split='train',  # All prepared data is currently in the 'train' folder
        transforms=transforms_obj
    )

    # Perform a random Train/Validation split (e.g., 90% Train, 10% Val)
    val_ratio = 0.1
    total_size = len(full_dataset)
    val_size = int(total_size * val_ratio)
    train_size = total_size - val_size

    generator = torch.Generator().manual_seed(42) # Fixed seed for reproducibility
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size], generator=generator
    )

    print(f"Split: {train_size} training samples, {val_size} validation samples")

except Exception as e:
    print(f"Error: Failed to load datasets: {e}")
    sys.exit(1)

# ------------------------------------------------------------
# Step 3: Create Data Loaders
# ------------------------------------------------------------
print("\nStep 3: Creating data loaders...")

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    collate_fn=collate_fn,
    persistent_workers=config.num_workers > 0
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")


# ------------------------------------------------------------
# Step 4: Setup Logging and Checkpointing
# ------------------------------------------------------------
print("\nStep 4: Setting up Weights & Biases...")
wandb.login()

logger = WandbLogger(
    project="eomt-binary-anomaly",
    name=f"anomaly-detection-{config.img_size[0]}",
    resume="allow"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=config.output_dir,
    monitor="val_loss",
    mode="min",
    save_top_k=3,
    every_n_epochs=1,
)

# ------------------------------------------------------------
# Step 5: Trainer Initialization
# ------------------------------------------------------------
print("\nStep 5: Setting up Lightning Trainer...")

trainer = L.Trainer(
    max_epochs=config.max_epochs,
    logger=logger,
    callbacks=[checkpoint_callback, LearningRateMonitor()],
    num_sanity_val_steps=0,
    precision="32",        # Using float32 for stability
    accelerator="gpu",
    devices="auto",        # Auto-detect available GPUs
    strategy="ddp" if torch.cuda.device_count() > 1 else "auto", # Distributed Data Parallel for multi-GPU
    gradient_clip_val=1.0, # Prevent gradient explosion
    log_every_n_steps=10,
    enable_progress_bar=True,
    sync_batchnorm=True if torch.cuda.device_count() > 1 else False,
)

# ------------------------------------------------------------
# Step 6: Start Training
# ------------------------------------------------------------
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Output shape:   [B, H, W] = [{config.batch_size}, {config.img_size[0]}, {config.img_size[1]}]")
print(f"Loss function:  Combined (Dice Loss + Focal Loss)")
print(f"Target values:  Binary masks (0=in-distribution, 1=anomaly)")
print(f"Encoder Status: FROZEN (preserving pre-trained Cityscapes features)")
print(f"Decoder Status: TRAINABLE (adapting to anomaly detection task)")
print("="*70 + "\n")

trainer.fit(model, train_loader, val_loader)

# ------------------------------------------------------------
# Completion
# ------------------------------------------------------------
print(f"\nSuccess: Training completed!")
print(f"Checkpoints saved to: {config.output_dir}")
print("\n" + "="*70)
print("TRAINING CONFIGURATION SUMMARY")
print("="*70)
print(f"Model Architecture: EoMT (Encoder-centric Mask Transformer)")
print(f"Pre-training:       Cityscapes semantic segmentation")
print(f"Task:               Binary anomaly detection (pixel-wise)")
print(f"Encoder:            FROZEN")
print(f"Decoder:            TRAINABLE")
print(f"Learning Rate:      {config.lr:.6f}")
print(f"Batch Size:         {config.batch_size}")
print(f"Image Size:         {config.img_size}")
print("="*70 + "\n")

BINARY ANOMALY DETECTION - FINE-TUNING PRE-TRAINED EOMT

Step 1: Loading pre-trained EoMT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Encoder frozen.
Ignoring pre-trained weights for: class_head.weight (Expected shape mismatch)
Ignoring pre-trained weights for: class_head.bias (Expected shape mismatch)
Success: Model loaded from /content/drive/MyDrive/Colab Notebooks/AML_Project/MaskArchitectureAnomaly_CourseProject/eomt/trained_model/epoch_106-step_19902_eomt.ckpt

Step 2: Loading prepared Outlier Exposure datasets...


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/custom_checkpoint_final_extension exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 93.5 M │ train │     0 │
│ 1 │ criterion │ BinaryAnomalyLoss      │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
│ 3 │ val_auroc │ BinaryAUROC            │      0 │ train │     0 │
│ 4 │ val_auprc │ BinaryAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 6.6 M                                                                                            
Non-trainable params: 86.9 M                                                                                       
Total params: 93.5 M                                                                                               
Total estimated model params size (MB): 373                                                                        
Modules in train mode: 302                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()